## IMPORT LIBRARY

In [1]:
import os
import time
import pyarrow as pa  # Opsional: Hanya untuk pamer/cek versi
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from xgboost.spark import SparkXGBClassifier 
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pyspark.sql.functions as F

## 1. INISIALISASI SPARK

In [ ]:
# Bersihkan sesi lama jika masih nyangkut
if 'spark' in locals():
    try: spark.stop()
    except: pass

# Gunakan nama file JAR yang WAJIB dipakai (sesuai arahan awal)
path_jar = "clickhouse-jdbc-0.6.3.jar"

# Pastikan file benar-benar ada sebelum Spark dijalankan
if not os.path.exists(path_jar):
    print(f"❌ AWAS: File JAR tidak ditemukan di '{path_jar}'!")
    print(f"Pastikan file {path_jar} ada di folder proyekmu.")
else:
    print("✅ File JAR ditemukan! Memulai SparkSession dalam Mode Maximum Memory (10GB)...")

    # Kita set Heap Memory 8GB + Off-Heap 2GB = Total 10GB RAM
    spark = SparkSession.builder \
        .appName("FlightDelayModelBuilder") \
        .config("spark.jars", path_jar) \
        .config("spark.driver.extraClassPath", path_jar) \
        .config("spark.executor.extraClassPath", path_jar) \
        .config("spark.executor.memory", "8g") \
        .config("spark.driver.memory", "8g") \
        .config("spark.driver.maxResultSize", "2g") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "2g") \
        .getOrCreate()

    print("✅ SparkSession lokal (Mode 10GB) berhasil terhubung!")

✅ File JAR ditemukan! Memulai SparkSession dalam Mode Maximum Memory (8GB)...
✅ SparkSession lokal (Mode 8GB) berhasil terhubung!


## 2. MEMBACA DATA DARI AWS

In [ ]:
CLICKHOUSE_URL = "jdbc:clickhouse://13.215.79.3:8123/flight_delay?compress=false&connection_timeout=120000&socket_timeout=120000&http_connection_provider=HTTP_URL_CONNECTION"

print("📥 Menarik data dari AWS dengan 31 Partisi (Berdasarkan Tanggal)...")

# Kita ganti partitionColumn menjadi DayofMonth (tanggal 1 s/d 31)
df_all = spark.read.format("jdbc") \
    .option("url", CLICKHOUSE_URL) \
    .option("dbtable", "ontime_features") \
    .option("user", "default") \
    .option("password", "rahasia123") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("partitionColumn", "DayofMonth") \
    .option("lowerBound", "1") \
    .option("upperBound", "31") \
    .option("numPartitions", "31") \
    .option("fetchsize", "10000") \
    .load()

print("✅ Skema data berhasil dimuat dengan 31 partisi mikro!")

📥 Menghubungkan dan menarik data menggunakan JDBC Partitioning...
✅ Peta skema data berhasil dimuat dengan 12 partisi paralel!
root
 |-- FlightDate: date (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- flight_year: integer (nullable = true)
 |-- flight_quarter: integer (nullable = true)
 |-- flight_month: integer (nullable = true)
 |-- flight_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- dep_hour: integer (nullable = true)
 |-- arr_hour: integer (nullable = true)
 |-- dep_time_bucket: string (nullable = true)
 |-- arr_time_bucket: string (nullable = true)
 |-- route: string (nullable = true)
 |-- same_state_route: integer (nullable = true)
 |-- distance_bucket: i

## 3. TIME-BASED SPLIT

In [4]:
import time
import os

print("📅 [STRATEGI INTEGRASI] Memulai Pemisahan Waktu & Unduh AWS...")

# Definisikan pemisahan dasar
df_train_all_partitions = df_all.filter(df_all.FlightDate < "2025-01-01")
df_test_raw_pull = df_all.filter(df_all.FlightDate >= "2025-01-01")

# Pastikan folder data_cache ada
os.makedirs("data_cache", exist_ok=True)

# ---------------------------------------------------------
# TAHAP A: TRAINING DATA (Pre-trigger & Simpan Lokal)
# ---------------------------------------------------------
print("\n📥 Tahap A: Mengunduh Data Training (2021-2024)...")
start_train = time.time()

# Pre-trigger per bulan agar tidak timeout
for month in range(1, 13):
    print(f"   ⏳ [Partisi {month}/12] Menyedot data penerbangan Bulan ke-{month:02d} dari AWS...")
    df_train_all_partitions.filter(df_train_all_partitions.flight_month == month).cache().count()

print("💾 Menyimpan Data Training ke Hard Disk lokal (Parquet)...")
df_train_all_partitions.write.mode("overwrite").parquet("data_cache/train_raw.parquet")
print(f"   ✔️ Sukses! Data Training aman di SSD. Durasi: {time.time() - start_train:.2f} detik.")

# ---------------------------------------------------------
# TAHAP B: TESTING DATA (Pre-trigger & Simpan Lokal)
# ---------------------------------------------------------
print("\n📥 Tahap B: Mengunduh Data Testing (Tahun 2025)...")
start_test = time.time()
df_test_raw_pull.cache().count()
print("💾 Menyimpan Data Testing ke Hard Disk lokal (Parquet)...")
df_test_raw_pull.write.mode("overwrite").parquet("data_cache/test_raw.parquet")
print(f"   ✔️ Sukses! Data Testing aman di SSD. Durasi: {time.time() - start_test:.2f} detik.")

# ---------------------------------------------------------
# TAHAP C: BACA KEMBALI DARI LOKAL (Bypass AWS)
# ---------------------------------------------------------
print("\n⚡ Memuat ulang data langsung dari SSD Lokal untuk proses selanjutnya...")
df_train_raw = spark.read.parquet("data_cache/train_raw.parquet")
df_test_raw = spark.read.parquet("data_cache/test_raw.parquet")

print("✅ Selesai! Pipa JDBC AWS ditutup. Komputasi ML selanjutnya 100% Offline & Kilat!")

📅 [STRATEGI INTEGRASI] Memulai Pemisahan Waktu & Unduh AWS...

📥 Tahap A: Mengunduh Data Training (2021-2024)...
   ⏳ [Partisi 1/12] Menyedot data penerbangan Bulan ke-01 dari AWS...
   ⏳ [Partisi 2/12] Menyedot data penerbangan Bulan ke-02 dari AWS...
   ⏳ [Partisi 3/12] Menyedot data penerbangan Bulan ke-03 dari AWS...
   ⏳ [Partisi 4/12] Menyedot data penerbangan Bulan ke-04 dari AWS...
   ⏳ [Partisi 5/12] Menyedot data penerbangan Bulan ke-05 dari AWS...
   ⏳ [Partisi 6/12] Menyedot data penerbangan Bulan ke-06 dari AWS...
   ⏳ [Partisi 7/12] Menyedot data penerbangan Bulan ke-07 dari AWS...
   ⏳ [Partisi 8/12] Menyedot data penerbangan Bulan ke-08 dari AWS...
   ⏳ [Partisi 9/12] Menyedot data penerbangan Bulan ke-09 dari AWS...
   ⏳ [Partisi 10/12] Menyedot data penerbangan Bulan ke-10 dari AWS...
   ⏳ [Partisi 11/12] Menyedot data penerbangan Bulan ke-11 dari AWS...
   ⏳ [Partisi 12/12] Menyedot data penerbangan Bulan ke-12 dari AWS...
💾 Menyimpan Data Training ke Hard Disk lokal

Py4JJavaError: An error occurred while calling o132.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 48.0 failed 1 times, most recent failure: Lost task 3.0 in stage 48.0 (TID 351) (Putik executor driver): java.sql.SQLException: java.io.IOException: Premature EOF
	at com.clickhouse.jdbc.SqlExceptionUtils.create(SqlExceptionUtils.java:45)
	at com.clickhouse.jdbc.SqlExceptionUtils.handle(SqlExceptionUtils.java:90)
	at com.clickhouse.jdbc.ClickHouseResultSet.next(ClickHouseResultSet.java:729)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNextWithoutTiming(JdbcUtils.scala:378)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.$anonfun$getNext$1(JdbcUtils.scala:396)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.withTimingNs(SQLMetrics.scala:234)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$$anon$1.getNext(JdbcUtils.scala:396)
	at org.apache.spark.sql.execution.datasou

## 4. BALANCING DATA SAMPLING (RANDOM UNDERSAMPLING)

In [ ]:
print("⚖️ Menjalankan proses balancing (Undersampling) terdistribusi pada data training...")

# Pisahkan kelas target
df_ontime = df_train_raw.filter(df_train_raw.arr_del15_label == 0)
df_delay = df_train_raw.filter(df_train_raw.arr_del15_label == 1)

count_ontime = df_ontime.count()
count_delay = df_delay.count()

# Hitung rasio fraksi untuk mengambil sampel acak data mayoritas (ontime)
fraction = count_delay / count_ontime

# Jalankan undersampling tanpa pengembalian (withReplacement=False)
df_ontime_sampled = df_ontime.sample(withReplacement=False, fraction=fraction, seed=42)

# Satukan kembali data training yang rasionya sudah seimbang 50:50
df_train_balanced = df_delay.union(df_ontime_sampled)

print(f"✅ Balancing selesai!")
print(f"📈 Total Data Training Setelah Diet Rasio: {df_train_balanced.count():,} baris.")

⚖️ Menjalankan proses balancing (Undersampling) terdistribusi pada data training...
✅ Balancing selesai!
📈 Total Data Training Setelah Diet Rasio: 2,068,130 baris.


## 5. FEATURE ENGINEERING & PREPROCESSING

In [ ]:
print("⚙️ Memulai transformasi fitur numerik dan kategorikal...")

# 1. Mengubah kolom teks kategorikal menjadi angka index (StringIndexer)
indexer_season = StringIndexer(inputCol="season", outputCol="season_index", handleInvalid="keep")
indexer_dist = StringIndexer(inputCol="distance_bucket", outputCol="distance_index", handleInvalid="keep")

# Terapkan transformasi indexer ke data
fit_season = indexer_season.fit(df_train_balanced)
fit_dist = indexer_dist.fit(df_train_balanced)

df_train_indexed = fit_dist.transform(fit_season.transform(df_train_balanced))
df_test_indexed = fit_dist.transform(fit_season.transform(df_test_raw))

# 2. Definisikan daftar fitur resmi yang akan disuapi ke model pohon
FEATURE_COLS = [
    'flight_month', 'dep_hour', 'is_weekend', 'same_state_route',
    'season_index', 'distance_index',
    'route_avg_arr_delay_prev', 'carrier_arr_delay_rate_prev'
]

# 3. Satukan seluruh array kolom ke dalam satu Vector tunggal 'features'
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")

df_train_final = assembler.transform(df_train_indexed).select("features", "arr_del15_label")
df_test_final = assembler.transform(df_test_indexed).select("features", "arr_del15_label")

print("✅ Pipeline persiapan fitur Spark MLlib siap digunakan!")

⚙️ Memulai transformasi fitur numerik dan kategorikal...
✅ Pipeline persiapan fitur Spark MLlib siap digunakan!


## 6. TRAINING MODEL

In [ ]:
print("🤖 Mempersiapkan inisialisasi 3 algoritma target...")

# Definisikan 3 model sesuai rencana awal kelompok
spark_models = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="arr_del15_label"),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="arr_del15_label", numTrees=100, seed=42),
    "XGBoost": SparkXGBClassifier(features_col="features", label_col="arr_del15_label", num_workers=1, eval_metric="logloss")
}

# Wadah untuk menyimpan model yang sudah terlatih
trained_spark_models = {}

print("\n🚀 Memulai proses training paralel di cluster Spark...")
for name, model in spark_models.items():
    print(f"   ⏳ Sedang melatih model: {name}...")
    trained_spark_models[name] = model.fit(df_train_final)
    print(f"   ✔️ {name} selesai dilatih!")

print("\n🎉 Semua 3 model berhasil dilatih dengan aman!")

🤖 Mempersiapkan inisialisasi 3 algoritma target...

🚀 Memulai proses training paralel di cluster Spark...
   ⏳ Sedang melatih model: Logistic Regression...
   ✔️ Logistic Regression selesai dilatih!
   ⏳ Sedang melatih model: Random Forest...


Py4JJavaError: An error occurred while calling o345.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 23 in stage 117.0 failed 1 times, most recent failure: Lost task 23.0 in stage 117.0 (TID 953) (Putik executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:64)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:363)
	at org.apache.spark.serializer.SerializerHelper$.$anonfun$serializeToChunkedBuffer$1(SerializerHelper.scala:40)
	at org.apache.spark.serializer.SerializerHelper$.$anonfun$serializeToChunkedBuffer$1$adapted(SerializerHelper.scala:40)
	at org.apache.spark.serializer.SerializerHelper$$$Lambda$3531/0x000001d8aa0a8200.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "d:\Pyta\Sem6\SBD\bigdata-final-project\.bigdata-venv\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "D:\anaconda3\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "d:\Pyta\Sem6\SBD\bigdata-final-project\.bigdata-venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "d:\Pyta\Sem6\SBD\bigdata-final-project\.bigdata-venv\Lib\site-packages\py4j\clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
        "Error while 

## 7. EVALUASI MODEL

In [ ]:
print("🕵️ Memulai proses evaluasi massal pada Data Testing (Data 2025)...")

# Penilai untuk metrik ROC-AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="arr_del15_label", rawPredictionCol="probability", metricName="areaUnderROC"
)

# List untuk menampung baris hasil evaluasi
final_eval_rows = []

for name, model in trained_spark_models.items():
    # Jalankan transformasi prediksi
    predictions = model.transform(df_test_final)
    
    # 1. Hitung skor ROC-AUC global
    roc_auc = evaluator_auc.evaluate(predictions)
    
    # 2. Hitung komponen performa biner (TP, TN, FP, FN) via Spark DataFrame
    prediction_and_labels = predictions.select("prediction", "arr_del15_label")
    
    tp = prediction_and_labels.filter((F.col("prediction") == 1.0) & (F.col("arr_del15_label") == 1)).count()
    tn = prediction_and_labels.filter((F.col("prediction") == 0.0) & (F.col("arr_del15_label") == 0)).count()
    fp = prediction_and_labels.filter((F.col("prediction") == 1.0) & (F.col("arr_del15_label") == 0)).count()
    fn = prediction_and_labels.filter((F.col("prediction") == 0.0) & (F.col("arr_del15_label") == 1)).count()
    
    # Kalkulasi rumus metrik evaluasi
    total_samples = tp + tn + fp + fn
    accuracy = (tp + tn) / total_samples if total_samples > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # Simpan ke list hasil
    final_eval_rows.append({
        "Algoritma": name,
        "Accuracy": f"{accuracy:.4f}",
        "Precision": f"{precision:.4f}",
        "Recall": f"{recall:.4f}",
        "F1-Score": f"{f1_score:.4f}",
        "ROC-AUC": f"{roc_auc:.4f}"
    })

# Cetak hasil akhir menggunakan bantuan Pandas DataFrame agar tercetak berupa tabel indah
import pandas as pd
df_spark_eval = pd.DataFrame(final_eval_rows)

print("\n📊 TABEL EVALUASI AKHIR PIPELINE APACHE SPARK (3 MODEL):")
print("=" * 75)
print(df_spark_eval.to_string(index=False))
print("=" * 75)

## 8. MENYIMPAN ARTIFACT MODEL

In [ ]:
print("💾 Menyimpan seluruh artifact model ke dalam folder models/...")

for name, model in trained_spark_models.items():
    folder_name = f"models/spark_{name.lower().replace(' ', '_')}_model"
    model.save(folder_name)
    print(f"   ✔️ Berhasil diexport: {folder_name}")

print("\n🎉 Mission Accomplished! Seluruh pengerjaan 3 model berbasis Spark selesai sempurna.")

In [ ]:
print("🆘 MENGAMANKAN DATA KE HARD DISK LOKAL...")

# Simpan data mentah yang sudah ditarik 1 jam tadi ke format Parquet
df_train_raw.write.mode("overwrite").parquet("data_cache/train_raw.parquet")
df_test_raw.write.mode("overwrite").parquet("data_cache/test_raw.parquet")

print("✅ Fiuh! Data aman. Sekarang kamu boleh me-restart kernel tanpa takut hilang!")

🆘 MENGAMANKAN DATA KE HARD DISK LOKAL...


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it